# Pandas from First Principles
## Notebook 9: Concatenating Data

---

### What You Will Learn

| # | Topic |
|---|-------|
| 1 | Why concatenation? concat vs merge |
| 2 | `pd.concat()` — stacking rows (axis=0) |
| 3 | `pd.concat()` — side by side (axis=1) |
| 4 | Handling mismatched columns |
| 5 | `keys` parameter — adding source labels |
| 6 | Concatenating a list of DataFrames |
| 7 | Common mistakes |
| 8 | Practice Questions & Answer Key |

> **Prerequisites:** Notebooks 1–8 (DataFrames, indexing, filtering, groupby, merging)

In [1]:
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 2.2.3


---
## 1. Introduction — Why Concatenation?

### The Real-World Problem

Imagine your company stores sales data in **separate monthly CSV files**:

```
january_sales.csv
february_sales.csv
march_sales.csv
...
```

To analyse the full year, you need to **stack** all these files into a single DataFrame. That's exactly what `pd.concat()` does.

---

### concat vs merge — What's the Difference?

| Feature | `pd.concat()` | `pd.merge()` |
|---------|--------------|-------------|
| **Purpose** | Stack / glue DataFrames together | Combine DataFrames on a shared key column |
| **Direction** | Vertically (rows) or horizontally (columns) | Horizontally (columns) based on a key |
| **Analogy** | Stacking LEGO bricks on top of each other | Zipping two lists together using a common ID |
| **Typical use** | Combining monthly/regional files | Joining orders with customer info |

**Simple mental model:**
- `concat` → *"I have more rows of the same kind of data"*  
- `merge` → *"I have related data I want to look up by key"*

In [2]:
# -------------------------------------------------------------------
# Dataset Setup — three monthly sales DataFrames used throughout
# -------------------------------------------------------------------

jan_sales = pd.DataFrame({
    'date':    ['2024-01-05', '2024-01-12', '2024-01-20'],
    'product': ['Laptop', 'Mouse', 'Keyboard'],
    'revenue': [75000, 1200, 2500],
    'region':  ['North', 'South', 'East'],
})

feb_sales = pd.DataFrame({
    'date':    ['2024-02-03', '2024-02-15', '2024-02-22'],
    'product': ['Monitor', 'Laptop', 'Webcam'],
    'revenue': [22000, 78000, 4500],
    'region':  ['East', 'North', 'South'],
})

mar_sales = pd.DataFrame({
    'date':     ['2024-03-08', '2024-03-19'],
    'product':  ['Phone', 'Tablet'],
    'revenue':  [28000, 35000],
    'region':   ['South', 'West'],
    'discount': [500, 1000],   # extra column not present in jan/feb
})

print("=== January Sales ===")
display(jan_sales)
print("\n=== February Sales ===")
display(feb_sales)
print("\n=== March Sales (has extra 'discount' column) ===")
display(mar_sales)

=== January Sales ===


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East



=== February Sales ===


,date,product,revenue,region
0,2024-02-03,Monitor,22000,East
1,2024-02-15,Laptop,78000,North
2,2024-02-22,Webcam,4500,South



=== March Sales (has extra 'discount' column) ===


,date,product,revenue,region,discount
0,2024-03-08,Phone,28000,South,500
1,2024-03-19,Tablet,35000,West,1000


---
## 2. `pd.concat()` — Stacking Rows (axis=0)

### What Is It?

`pd.concat()` glues multiple DataFrames together along an axis.  
With `axis=0` (the default), it **stacks rows** — like piling sheets of paper on top of each other.

### Analogy

Think of three paper notebooks — January, February, March.  
`pd.concat()` with `axis=0` is like **stapling them together** so you get one long notebook.

### Syntax

```python
pd.concat(
    objs,             # list of DataFrames to combine
    axis=0,           # 0 = stack rows, 1 = stack columns
    ignore_index=False,  # True → reset to 0,1,2... index
    keys=None,        # labels to identify each source DataFrame
    join='outer'      # 'outer' keeps all columns, 'inner' keeps only common ones
)
```

### Parameters

| Parameter | Default | What it does |
|-----------|---------|-------------|
| `objs` | — | **Required.** A list (or dict) of DataFrames/Series to concatenate |
| `axis` | `0` | `0` = stack rows; `1` = stack columns side by side |
| `ignore_index` | `False` | `True` = produces a fresh `0, 1, 2 ...` index after stacking |
| `keys` | `None` | Adds an outer level to the index so you know which DataFrame each row came from |
| `join` | `'outer'` | `'outer'` fills missing columns with NaN; `'inner'` keeps only shared columns |

### Return Value

Returns a **new DataFrame** — the original DataFrames are never modified.

In [ ]:
# -------------------------------------------------------------------
# Basic row-stacking: concat January + February
# -------------------------------------------------------------------

combined_basic = pd.concat([jan_sales, feb_sales],axis=0)

print("Combined January + February (default settings):")
display(combined_basic)
print("\nShape:", combined_basic.shape)
print("\nNotice the index: 0,1,2, 0,1,2 — duplicate labels!")
print("Index values:", combined_basic.index.tolist())

Combined January + February (default settings):


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East
0,2024-02-03,Monitor,22000,East
1,2024-02-15,Laptop,78000,North
2,2024-02-22,Webcam,4500,South



Shape: (6, 4)

Notice the index: 0,1,2, 0,1,2 — duplicate labels!
Index values: [0, 1, 2, 0, 1, 2]


### The Duplicate Index Problem

After a basic concat, the original index labels are **preserved**.  
This means both January and February contribute rows `0, 1, 2` — giving you **duplicate index values**.

Duplicate index values can cause subtle bugs when you slice with `.loc[]`.  
The fix: `ignore_index=True`.

In [9]:
# -------------------------------------------------------------------
# Fix: ignore_index=True for a clean, sequential index
# -------------------------------------------------------------------

combined_clean = pd.concat([jan_sales, feb_sales], ignore_index=True)

print("Combined with ignore_index=True:")
display(combined_clean)
print("\nIndex values now:", combined_clean.index.tolist())
print("✓ Clean sequential index: 0, 1, 2, 3, 4, 5")

Combined with ignore_index=True:


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East
3,2024-02-03,Monitor,22000,East
4,2024-02-15,Laptop,78000,North
5,2024-02-22,Webcam,4500,South



Index values now: [0, 1, 2, 3, 4, 5]
✓ Clean sequential index: 0, 1, 2, 3, 4, 5


---
## 3. `pd.concat()` — Side by Side (axis=1)

### What Is It?

With `axis=1`, `pd.concat()` places DataFrames **side by side**, adding new columns.  
Instead of stacking rows, it extends the table to the right.

### Analogy

Imagine two spreadsheets open side by side — `axis=1` is like **copy-pasting columns** from the second sheet next to the first.

### What Happens When Row Counts Differ?

If one DataFrame has more rows than another, the shorter one gets **NaN** in the extra rows.  
Pandas aligns on the index, so matching index labels share a row; non-matching ones produce NaN.

In [10]:
# -------------------------------------------------------------------
# axis=1: place two DataFrames side by side
# -------------------------------------------------------------------

side_by_side = pd.concat([jan_sales, feb_sales], axis=1)

print("January and February placed side by side (axis=1):")
display(side_by_side)
print("\nShape:", side_by_side.shape)
print("Notice: columns are doubled (date, product, revenue, region appear twice)")

January and February placed side by side (axis=1):


,date,product,revenue,region,date,product,revenue,region
0,2024-01-05,Laptop,75000,North,2024-02-03,Monitor,22000,East
1,2024-01-12,Mouse,1200,South,2024-02-15,Laptop,78000,North
2,2024-01-20,Keyboard,2500,East,2024-02-22,Webcam,4500,South



Shape: (3, 8)
Notice: columns are doubled (date, product, revenue, region appear twice)


In [11]:
# -------------------------------------------------------------------
# Practical axis=1 use: adding a new column block from another source
# -------------------------------------------------------------------

# Imagine we have a separate DataFrame with discount info for January
jan_discounts = pd.DataFrame({
    'discount': [200, 50, 100],
    'coupon':   ['SAVE200', 'SAVE50', 'SAVE100'],
})

jan_with_discounts = pd.concat([jan_sales, jan_discounts], axis=1)

print("January sales with discount columns added (axis=1):")
display(jan_with_discounts)

January sales with discount columns added (axis=1):


,date,product,revenue,region,discount,coupon
0,2024-01-05,Laptop,75000,North,200,SAVE200
1,2024-01-12,Mouse,1200,South,50,SAVE50
2,2024-01-20,Keyboard,2500,East,100,SAVE100


In [ ]:
# -------------------------------------------------------------------
# axis=1 when row counts differ — NaN fills the gaps
# -------------------------------------------------------------------

# jan_sales has 3 rows, mar_sales has 2 rows
unequal_concat = pd.concat([jan_sales, mar_sales], axis=1)

print("Concatenating DataFrames with different row counts (axis=1):")
display(unequal_concat)
print("\nNaN appears for rows that don't exist in the shorter DataFrame.")

Concatenating DataFrames with different row counts (axis=1):


,date,product,revenue,region,date,product,revenue,region,discount
0,2024-01-05,Laptop,75000,North,2024-03-08,Phone,28000.0,South,500.0
1,2024-01-12,Mouse,1200,South,2024-03-19,Tablet,35000.0,West,1000.0
2,2024-01-20,Keyboard,2500,East,NaN,NaN,NaN,NaN,NaN



NaN appears for rows that don't exist in the shorter DataFrame.


---
## 4. Handling Mismatched Columns

### The Problem

Real-world data is messy. Monthly files might not have **identical columns**.  
Here, `mar_sales` has an extra `discount` column that `jan_sales` and `feb_sales` do not have.

### What Happens by Default (`join='outer'`)?

Pandas keeps **all columns** from all DataFrames.  
Rows from DataFrames that were missing a column get `NaN` for it.

### Alternative: `join='inner'`

`join='inner'` keeps **only the columns that exist in every DataFrame**.  
This drops any column that doesn't appear in all sources.

| | `join='outer'` | `join='inner'` |
|-|---------------|---------------|
| **Keeps** | All columns | Only shared columns |
| **Missing data** | Filled with NaN | No NaN from column mismatch |
| **Use when** | You want to preserve everything | You want a clean, uniform table |

In [15]:
# -------------------------------------------------------------------
# join='outer' (default): keep all columns, NaN for missing values
# -------------------------------------------------------------------

all_months_outer = pd.concat(
    [jan_sales, feb_sales, mar_sales],
    ignore_index=True,
    join='outer'   # this is the default — shown explicitly for clarity
)

print("All months concatenated with join='outer' (default):")
display(all_months_outer)
print("\njan & feb rows show NaN in 'discount' column — they had no discount data.")

All months concatenated with join='outer' (default):


,date,product,revenue,region,discount
0,2024-01-05,Laptop,75000,North,NaN
1,2024-01-12,Mouse,1200,South,NaN
2,2024-01-20,Keyboard,2500,East,NaN
3,2024-02-03,Monitor,22000,East,NaN
4,2024-02-15,Laptop,78000,North,NaN
5,2024-02-22,Webcam,4500,South,NaN
6,2024-03-08,Phone,28000,South,500.0
7,2024-03-19,Tablet,35000,West,1000.0



jan & feb rows show NaN in 'discount' column — they had no discount data.


In [16]:
# -------------------------------------------------------------------
# join='inner': keep only columns common to ALL DataFrames
# -------------------------------------------------------------------

all_months_inner = pd.concat(
    [jan_sales, feb_sales, mar_sales],
    ignore_index=True,
    join='inner'   # only columns present in jan AND feb AND mar
)

print("All months concatenated with join='inner':")
display(all_months_inner)
print("\nColumns kept:", all_months_inner.columns.tolist())
print("'discount' column is gone — it wasn't in jan_sales or feb_sales.")

All months concatenated with join='inner':


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East
3,2024-02-03,Monitor,22000,East
4,2024-02-15,Laptop,78000,North
5,2024-02-22,Webcam,4500,South
6,2024-03-08,Phone,28000,South
7,2024-03-19,Tablet,35000,West



Columns kept: ['date', 'product', 'revenue', 'region']
'discount' column is gone — it wasn't in jan_sales or feb_sales.


---
## 5. The `keys` Parameter — Adding Source Labels

### What Is It?

The `keys` parameter lets you **tag each DataFrame** with a label before concatenating.  
This creates a **MultiIndex** so you always know which source a row came from.

### Why Is It Useful?

After combining many DataFrames, it's easy to lose track of where each row originated.  
The `keys` parameter solves this by adding an outer index level with a meaningful label.

### Analogy

Imagine stapling three reports together and writing **"January"**, **"February"**, **"March"**  
on a sticky note at the start of each section. `keys` does exactly that — programmatically.

### Flattening with `reset_index()`

A MultiIndex can be tricky to work with. Use `.reset_index()` to turn the outer index level  
into a regular column — giving you a clean, flat DataFrame with a `month` (or similar) column.

In [18]:
# -------------------------------------------------------------------
# keys parameter: label each source DataFrame
# -------------------------------------------------------------------

combined_with_keys = pd.concat(
    [jan_sales, feb_sales],
    keys=['January', 'February']
)

print("Concatenated with keys — notice the MultiIndex:")
display(combined_with_keys)
print("\nIndex type:", type(combined_with_keys.index))

Concatenated with keys — notice the MultiIndex:


date   product  revenue region
January  0  2024-01-05    Laptop    75000  North
         1  2024-01-12     Mouse     1200  South
         2  2024-01-20  Keyboard     2500   East
February 0  2024-02-03   Monitor    22000   East
         1  2024-02-15    Laptop    78000  North
         2  2024-02-22    Webcam     4500  South


Index type: <class 'pandas.core.indexes.multi.MultiIndex'>


In [19]:
# -------------------------------------------------------------------
# Flatten the MultiIndex with reset_index()
# -------------------------------------------------------------------

combined_flat = pd.concat(
    [jan_sales, feb_sales],
    keys=['January', 'February']
).reset_index(level=0, names=['month']).reset_index(drop=True)

print("Flattened result — 'month' column tells us the source:")
display(combined_flat)
print("\n'month' column makes it easy to filter: combined_flat[combined_flat['month']=='January']")

Flattened result — 'month' column tells us the source:


,month,date,product,revenue,region
0,January,2024-01-05,Laptop,75000,North
1,January,2024-01-12,Mouse,1200,South
2,January,2024-01-20,Keyboard,2500,East
3,February,2024-02-03,Monitor,22000,East
4,February,2024-02-15,Laptop,78000,North
5,February,2024-02-22,Webcam,4500,South



'month' column makes it easy to filter: combined_flat[combined_flat['month']=='January']


In [20]:
# -------------------------------------------------------------------
# Simpler approach: add a 'month' column before concatenating
# (often cleaner than dealing with MultiIndex)
# -------------------------------------------------------------------

jan_tagged = jan_sales.copy()
feb_tagged = feb_sales.copy()

jan_tagged['month'] = 'January'
feb_tagged['month'] = 'February'

combined_tagged = pd.concat([jan_tagged, feb_tagged], ignore_index=True)

print("Alternative: tag before concatenating — clean and straightforward:")
print(combined_tagged)

Alternative: tag before concatenating — clean and straightforward:
         date   product  revenue region     month
0  2024-01-05    Laptop    75000  North   January
1  2024-01-12     Mouse     1200  South   January
2  2024-01-20  Keyboard     2500   East   January
3  2024-02-03   Monitor    22000   East  February
4  2024-02-15    Laptop    78000  North  February
5  2024-02-22    Webcam     4500  South  February


---
## 6. Concatenating a List of DataFrames

### The Pattern

`pd.concat()` accepts **any iterable** of DataFrames — including a Python list.  
This is powerful because you can build that list dynamically in a loop.

### The Real-World Loop Pattern

```python
import glob

all_frames = []
for filepath in glob.glob('data/sales_*.csv'):
    df = pd.read_csv(filepath)
    all_frames.append(df)

full_data = pd.concat(all_frames, ignore_index=True)
```

This is one of the **most common patterns** in real data engineering pipelines.

> ⚠️ **Performance tip:** Always collect DataFrames in a list first, then call `pd.concat()` once.  
> Calling `pd.concat()` inside a loop (concatenating one row at a time) is very slow — it creates a new DataFrame at every iteration.

In [22]:
# -------------------------------------------------------------------
# Concatenating a list of DataFrames — four months in one go
# -------------------------------------------------------------------

# Simulate a 4th month
apr_sales = pd.DataFrame({
    'date':    ['2024-04-02', '2024-04-17'],
    'product': ['Headphones', 'Charger'],
    'revenue': [8500, 1800],
    'region':  ['North', 'East'],
})

# Pass all four as a list
all_months = pd.concat(
    [jan_sales, feb_sales, apr_sales],
    ignore_index=True
)

print("Three months concatenated from a list:")
display(all_months)
print("\nTotal rows:", len(all_months))

Three months concatenated from a list:


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East
3,2024-02-03,Monitor,22000,East
4,2024-02-15,Laptop,78000,North
5,2024-02-22,Webcam,4500,South
6,2024-04-02,Headphones,8500,North
7,2024-04-17,Charger,1800,East



Total rows: 8


In [24]:
# -------------------------------------------------------------------
# Real-world pattern: simulate reading files in a loop
# -------------------------------------------------------------------

# Simulate a dict of 'filenames' -> DataFrames (normally you'd use pd.read_csv)
monthly_data = {
    'january.csv':  jan_sales,
    'february.csv': feb_sales,
    'april.csv':    apr_sales,
}

all_frames = []   # step 1: collect all DataFrames in a list

for filename, df in monthly_data.items():
    temp = df.copy()
    temp['source_file'] = filename   # tag the source so we know where each row came from
    all_frames.append(temp)
    print(f"Loaded: {filename}  ({len(temp)} rows)")

# step 2: concat ONCE at the end (not inside the loop)
full_data = pd.concat(all_frames, ignore_index=True)

print("\nFull dataset after loop-and-concat:")
display(full_data)
print("\nTotal rows:", len(full_data))

Loaded: january.csv  (3 rows)
Loaded: february.csv  (3 rows)
Loaded: april.csv  (2 rows)

Full dataset after loop-and-concat:


,date,product,revenue,region,source_file
0,2024-01-05,Laptop,75000,North,january.csv
1,2024-01-12,Mouse,1200,South,january.csv
2,2024-01-20,Keyboard,2500,East,january.csv
3,2024-02-03,Monitor,22000,East,february.csv
4,2024-02-15,Laptop,78000,North,february.csv
5,2024-02-22,Webcam,4500,South,february.csv
6,2024-04-02,Headphones,8500,North,april.csv
7,2024-04-17,Charger,1800,East,april.csv



Total rows: 8


---
## 7. Common Mistakes

### Mistake 1: Using `+` Instead of `pd.concat()`

The `+` operator on DataFrames performs **element-wise addition**, not stacking.  
It will try to add numeric columns together and usually raises a `TypeError` or gives wrong results.

```python
# ❌ WRONG
combined = jan_sales + feb_sales   # This adds values, not rows!

# ✅ CORRECT
combined = pd.concat([jan_sales, feb_sales], ignore_index=True)
```

---

### Mistake 2: Forgetting `ignore_index=True`

After a basic concat, both DataFrames keep their original index labels.  
This creates **duplicate index values** that can cause silent bugs:

```python
# ❌ Duplicate index — can cause confusion
combined = pd.concat([jan_sales, feb_sales])
# Index: 0, 1, 2, 0, 1, 2

# ✅ Clean sequential index
combined = pd.concat([jan_sales, feb_sales], ignore_index=True)
# Index: 0, 1, 2, 3, 4, 5
```

---

### Mistake 3: Confusing `axis=0` and `axis=1`

| | `axis=0` | `axis=1` |
|-|----------|----------|
| **Direction** | Down ↓ (more rows) | Right → (more columns) |
| **Think of it as** | Stacking bricks vertically | Placing bricks side by side |
| **Common use** | Combining monthly files | Adding new feature columns |

**Memory trick:** axis=**0** → grows the **0th dimension** (rows).  
axis=**1** → grows the **1st dimension** (columns).

In [ ]:
# -------------------------------------------------------------------
# Demonstrating common mistake: + operator on DataFrames
# -------------------------------------------------------------------

print("What '+' actually does to DataFrames:")
try:
    wrong = jan_sales + feb_sales
    print(wrong)
    print("\n^ Notice: only numeric columns are added; strings produce NaN — NOT a row stack!")
except Exception as e:
    print("Error:", e)

print("\n---")
print("Correct way to stack rows:")
correct = pd.concat([jan_sales, feb_sales], ignore_index=True)
print(correct)

In [26]:
# -------------------------------------------------------------------
# Demonstrating duplicate index problem and the fix
# -------------------------------------------------------------------

combined_no_reset = pd.concat([jan_sales, feb_sales])   # duplicate index!
combined_with_reset = pd.concat([jan_sales, feb_sales], ignore_index=True)

print("WITHOUT ignore_index:")
print("Index:", combined_no_reset.index.tolist())
print("Calling .loc[0] returns MULTIPLE rows — both Jan and Feb index-0 rows:")
display(combined_no_reset.loc[0])

print("\nWITH ignore_index=True:")
print("Index:", combined_with_reset.index.tolist())
print(".loc[0] returns exactly ONE row:")
display(combined_with_reset.loc[0])

WITHOUT ignore_index:
Index: [0, 1, 2, 0, 1, 2]
Calling .loc[0] returns MULTIPLE rows — both Jan and Feb index-0 rows:


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
0,2024-02-03,Monitor,22000,East



WITH ignore_index=True:
Index: [0, 1, 2, 3, 4, 5]
.loc[0] returns exactly ONE row:


date       2024-01-05
product        Laptop
revenue         75000
region          North
Name: 0, dtype: object

---
## 8. Practice Questions

Work through these questions on your own, then check the answer key below.

| # | Difficulty | Topic |
|---|-----------|-------|
| Q1 | 🟢 Easy | Concat two DataFrames, reset index |
| Q2 | 🟢 Easy | Concat all three months, observe NaN |
| Q3 | 🟢 Easy | Concat side by side (axis=1) |
| Q4 | 🟢 Easy | Concat with keys and reset_index |
| Q5 | 🟡 Medium | Concat with join='inner' |
| Q6 | 🟡 Medium | Loop pattern: collect in list, concat once |
| Q7 | 🟡 Medium | Aggregate after concat |

---

### Q1 🟢 — Concatenate `jan_sales` and `feb_sales`, reset the index
Stack `jan_sales` and `feb_sales` into one DataFrame.  
Make sure the resulting index is `0, 1, 2, 3, 4, 5`.

In [30]:
pd.concat([jan_sales,feb_sales],
          ignore_index=True)


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East
3,2024-02-03,Monitor,22000,East
4,2024-02-15,Laptop,78000,North
5,2024-02-22,Webcam,4500,South


### Q2 🟢 — Concatenate all three months; observe the `discount` column
Concatenate `jan_sales`, `feb_sales`, and `mar_sales` with `ignore_index=True`.  
Print the result and note what value appears in the `discount` column for January and February rows.

In [31]:
pd.concat([jan_sales,feb_sales,mar_sales],
          ignore_index=True
          )


,date,product,revenue,region,discount
0,2024-01-05,Laptop,75000,North,NaN
1,2024-01-12,Mouse,1200,South,NaN
2,2024-01-20,Keyboard,2500,East,NaN
3,2024-02-03,Monitor,22000,East,NaN
4,2024-02-15,Laptop,78000,North,NaN
5,2024-02-22,Webcam,4500,South,NaN
6,2024-03-08,Phone,28000,South,500.0
7,2024-03-19,Tablet,35000,West,1000.0


### Q3 🟢 — Concatenate `jan_sales` and `feb_sales` side by side (`axis=1`)
What shape does the result have? How many columns are there?

In [33]:
pd.concat([jan_sales,feb_sales,],
          axis=1)


,date,product,revenue,region,date,product,revenue,region
0,2024-01-05,Laptop,75000,North,2024-02-03,Monitor,22000,East
1,2024-01-12,Mouse,1200,South,2024-02-15,Laptop,78000,North
2,2024-01-20,Keyboard,2500,East,2024-02-22,Webcam,4500,South


### Q4 🟢 — Concatenate `jan_sales` and `feb_sales` with `keys=['January','February']`, then flatten
Use `keys` to label each source. Then call `.reset_index()` to get a flat DataFrame  
where the first column identifies the month.

In [36]:
pd.concat([jan_sales,feb_sales],
          keys=['January','February']).reset_index()


,level_0,level_1,date,product,revenue,region
0,January,0,2024-01-05,Laptop,75000,North
1,January,1,2024-01-12,Mouse,1200,South
2,January,2,2024-01-20,Keyboard,2500,East
3,February,0,2024-02-03,Monitor,22000,East
4,February,1,2024-02-15,Laptop,78000,North
5,February,2,2024-02-22,Webcam,4500,South


### Q5 🟡 — Concatenate all three months keeping only common columns (`join='inner'`)
Concatenate `jan_sales`, `feb_sales`, and `mar_sales` with `join='inner'`.  
Which columns survive? What happened to `discount`?

In [37]:
pd.concat([jan_sales,feb_sales,mar_sales],
          join='inner')


,date,product,revenue,region
0,2024-01-05,Laptop,75000,North
1,2024-01-12,Mouse,1200,South
2,2024-01-20,Keyboard,2500,East
0,2024-02-03,Monitor,22000,East
1,2024-02-15,Laptop,78000,North
2,2024-02-22,Webcam,4500,South
0,2024-03-08,Phone,28000,South
1,2024-03-19,Tablet,35000,West


### Q6 🟡 — Simulate the loop pattern
You have three monthly DataFrames. Write a loop that:
1. Tags each DataFrame with a `'month'` column (`'Jan'`, `'Feb'`, `'Mar'`)
2. Appends each to a list
3. Calls `pd.concat()` once at the end

Print the final DataFrame and verify the `'month'` column is present.

In [ ]:
# Q6 — Your answer here


### Q7 🟡 — Total revenue by product across all months
Concatenate all three months (`jan_sales`, `feb_sales`, `mar_sales`) with `ignore_index=True`.  
Then use `.groupby()` to find the **total revenue per product**, sorted from highest to lowest.

In [46]:
df = pd.concat([jan_sales,feb_sales,mar_sales],ignore_index=True)
display(df.groupby('product')['revenue'].sum().sort_values(ascending=False).reset_index())


,product,revenue
0,Laptop,153000
1,Tablet,35000
2,Phone,28000
3,Monitor,22000
4,Webcam,4500
5,Keyboard,2500
6,Mouse,1200


---
## Answer Key

### A1 — Concatenate `jan_sales` and `feb_sales`, reset the index

In [ ]:
# A1
q1_result = pd.concat([jan_sales, feb_sales], ignore_index=True)

print("Jan + Feb concatenated with clean index:")
print(q1_result)
print("\nIndex:", q1_result.index.tolist())

### A2 — Concatenate all three months; observe `discount`

In [ ]:
# A2
q2_result = pd.concat([jan_sales, feb_sales, mar_sales], ignore_index=True)

print("All three months concatenated:")
print(q2_result)
print("\nJan and Feb rows have NaN in 'discount' — they had no such column.")
print("Mar rows have actual discount values: 500 and 1000.")

### A3 — Concatenate side by side (`axis=1`)

In [ ]:
# A3
q3_result = pd.concat([jan_sales, feb_sales], axis=1)

print("Jan and Feb side by side:")
print(q3_result)
print("\nShape:", q3_result.shape)
print("Columns:", q3_result.columns.tolist())
print("\nResult has", q3_result.shape[0], "rows and", q3_result.shape[1], "columns.")
print("Column names are repeated because both DataFrames had identical column names.")

### A4 — Concatenate with `keys`, then flatten with `reset_index()`

In [ ]:
# A4
q4_result = (
    pd.concat([jan_sales, feb_sales], keys=['January', 'February'])
    .reset_index(level=0, names=['month'])
    .reset_index(drop=True)
)

print("Concatenated with keys, then flattened:")
print(q4_result)
print("\n'month' column identifies the original source of each row.")

### A5 — Concatenate with `join='inner'` (common columns only)

In [ ]:
# A5
q5_result = pd.concat(
    [jan_sales, feb_sales, mar_sales],
    ignore_index=True,
    join='inner'
)

print("All months with join='inner':")
print(q5_result)
print("\nColumns:", q5_result.columns.tolist())
print("\n'discount' is gone — it was only in mar_sales, so it's excluded by inner join.")
print("Surviving columns: date, product, revenue, region (present in ALL three DataFrames).")

### A6 — Simulate the loop pattern

In [ ]:
# A6
monthly_frames = [
    (jan_sales, 'Jan'),
    (feb_sales, 'Feb'),
    (mar_sales, 'Mar'),
]

all_frames = []   # collect here — DO NOT concat inside the loop

for df, month_name in monthly_frames:
    tagged = df.copy()
    tagged['month'] = month_name
    all_frames.append(tagged)
    print(f"Tagged {month_name}: {len(tagged)} rows")

# concat once, outside the loop
q6_result = pd.concat(all_frames, ignore_index=True)

print("\nFinal concatenated DataFrame:")
print(q6_result)
print("\n'month' column is present:", 'month' in q6_result.columns)

### A7 — Total revenue by product across all months

In [44]:
# A7
all_data = pd.concat([jan_sales, feb_sales, mar_sales], ignore_index=True)

revenue_by_product = (
    all_data
    .groupby('product')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

revenue_by_product.columns = ['product', 'total_revenue']

print("Total revenue by product (all months combined):")
display(revenue_by_product)
print("\nLaptop appears in both Jan and Feb — revenues are summed correctly.")

Total revenue by product (all months combined):


,product,total_revenue
0,Laptop,153000
1,Tablet,35000
2,Phone,28000
3,Monitor,22000
4,Webcam,4500
5,Keyboard,2500
6,Mouse,1200



Laptop appears in both Jan and Feb — revenues are summed correctly.


---
## Quick Revision Table

| Concept | Code Pattern | Key Point |
|---------|-------------|----------|
| Stack rows | `pd.concat([df1, df2], ignore_index=True)` | Always use `ignore_index=True` to avoid duplicate index labels |
| Stack columns | `pd.concat([df1, df2], axis=1)` | Aligns on index; shorter DataFrame gets NaN in extra rows |
| Keep all columns | `join='outer'` (default) | Missing columns filled with NaN |
| Keep common columns only | `join='inner'` | Drops any column not present in every DataFrame |
| Label sources | `keys=['Jan', 'Feb']` | Creates MultiIndex; flatten with `.reset_index()` |
| Loop pattern | Append to list → `pd.concat(list)` once | Never concat inside a loop — it's slow |
| Fix duplicate index | `ignore_index=True` | Produces clean 0,1,2... index after stacking |
| concat vs merge | `concat` → more rows/columns of same type | `merge` → lookup by shared key column |

---

## 3 Interview Questions

**Q1.** What is the difference between `pd.concat()` and `pd.merge()`?  
**A.** `pd.concat()` stacks DataFrames along an axis (rows or columns) without needing a shared key. `pd.merge()` combines DataFrames based on the values in one or more shared key columns (like a SQL JOIN). Use `concat` when you have more rows/columns of the same kind of data; use `merge` when you're relating two different datasets by a common identifier.

---

**Q2.** What happens when you concatenate two DataFrames that have different columns? How do `join='outer'` and `join='inner'` behave differently?  
**A.** With `join='outer'` (the default), the result keeps every column from all DataFrames. Rows from DataFrames that were missing a column get `NaN` for that column. With `join='inner'`, only columns that appear in **all** DataFrames are kept — no NaN from column mismatch, but you may lose data from columns unique to one source.

---

**Q3.** Why is it a bad practice to call `pd.concat()` inside a loop on every iteration? What is the recommended pattern?  
**A.** Calling `pd.concat()` inside a loop creates a brand-new DataFrame on every iteration by copying all the data seen so far. This leads to O(n²) memory copies and becomes very slow for large datasets. The correct pattern is to **append each DataFrame to a Python list** inside the loop, then call `pd.concat(list)` exactly **once** after the loop finishes. This is O(n) and dramatically faster.

---

## What's Next

**Notebook 10 — Reshaping Data**

You've mastered concatenation — stacking DataFrames together. Next, we'll explore how to **reshape** a DataFrame's structure:

| Topic | Preview |
|-------|---------|
| `df.melt()` | Convert wide format → long format |
| `df.pivot()` | Convert long format → wide format |
| `df.pivot_table()` | Pivot with aggregation (like Excel pivot tables) |
| `df.stack()` / `df.unstack()` | Reshape MultiIndex DataFrames |

These are essential tools for transforming data into the shape that your analysis or visualisation library needs.